# 03 — Splits construction

This notebook constructs the four canonical splits used in the audit, **derives them from the participants manifest and the on-disk audio**, and writes a single participant-level mask file to `data/metadata/splits.csv`. It also documents exactly *who* is dropped at each filtering stage and *why* — both as an audit trail and as a sanity check that the rules described in notebook 02 produce the headline numbers reported in the paper.

**The four splits**

| Split | Filter | Participants | Clips | Use |
|---|---|---:|---:|---|
| `full` | none | 515 | 20,008 | Everything we collected (release-level superset). |
| `audit_eligible` | `race_simplified ∈ {Black, White}` ∧ `gender_simplified ∈ {Female, Male}` | 506 | 19,680 | Broad fold; any participant fitting the 2×2 audit grid by self-report. |
| `strict_audit` | `audit_eligible` ∧ `prolific_filter_matches_self_report` | 500 | 19,449 | Conservative fold; recommended for primary disparity analyses. |
| `balanced` | `strict_audit`, then uniformly sampled to 124 participants per (race × gender) cell (seed=42) | 496 | 19,304 | Equal-cell fold; used wherever cell-size imbalance would distort an estimator. |

**Output artifact**

We chose a *participant-level mask* representation (rather than four separate clip-level CSVs) because:

1. The clip → participant mapping is trivially recoverable from the audio path (`data/audio/{cohort}/{participant_id}/{participant_id}_{q##}.wav`), so a per-clip split would mostly duplicate information.
2. A single 515-row file is easier to keep consistent than four ~20k-row files.
3. Clip-level subsets can be reconstructed on the fly with a one-liner (shown at the end of the notebook).

`splits.csv` has 515 rows and the following columns:

- `participant_id`
- `in_full` (always `True` — included for symmetry)
- `in_audit_eligible`
- `in_strict_audit`
- `in_balanced`

**Inputs:** `../data/metadata/participants.csv`, `../data/audio/`  
**Outputs (written by this notebook):** `../data/metadata/splits.csv`

## Setup

In [1]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent
PARTICIPANTS_CSV = REPO_ROOT / "data" / "metadata" / "participants.csv"
AUDIO_DIR = REPO_ROOT / "data" / "audio"
SPLITS_CSV = REPO_ROOT / "data" / "metadata" / "splits.csv"

BALANCED_SEED = 42
AUDIT_RACES = {"Black", "White"}
AUDIT_GENDERS = {"Female", "Male"}

participants = pd.read_csv(PARTICIPANTS_CSV)
print(f"Loaded {len(participants)} participants.")

Loaded 515 participants.


### Enumerate clips on disk

Rather than depending on a separate clip manifest, we walk `data/audio/{cohort}/{participant_id}/*.wav` directly. This makes the notebook self-validating: if a participant's audio folder is missing or partially populated, the clip count will show it immediately.

In [2]:
clips_per_pid = {
    pid_dir.name: len(list(pid_dir.glob("*.wav")))
    for cohort_dir in AUDIO_DIR.iterdir() if cohort_dir.is_dir()
    for pid_dir in cohort_dir.iterdir() if pid_dir.is_dir()
}
clips_per_pid = pd.Series(clips_per_pid, name="n_clips").rename_axis("participant_id")
total_clips = int(clips_per_pid.sum())
print(f"{len(clips_per_pid)} participants with audio; {total_clips} clips total.")

missing = set(participants["participant_id"]) - set(clips_per_pid.index)
orphaned = set(clips_per_pid.index) - set(participants["participant_id"])
assert not missing, f"Participants in participants.csv but no audio folder: {missing}"
assert not orphaned, f"Audio folders with no row in participants.csv: {orphaned}"

participants = participants.merge(clips_per_pid.reset_index(), on="participant_id")

515 participants with audio; 20008 clips total.


## 1. The `full` split

The trivial fold: every participant we collected, every clip on disk. We surface it as an explicit split so downstream code can always reference splits by name.

In [3]:
in_full = pd.Series(True, index=participants.index)
n_full = int(in_full.sum())
clips_full = int(participants.loc[in_full, "n_clips"].sum())
print(f"full: {n_full} participants  |  {clips_full} clips")

full: 515 participants  |  20008 clips


## 2. The `audit_eligible` split

**Rule:** keep participants whose self-report falls inside the Black/White × Female/Male grid:

$$\texttt{in\_audit\_eligible} \iff \texttt{race\_simplified} \in \{\text{Black}, \text{White}\} \wedge \texttt{gender\_simplified} \in \{\text{Female}, \text{Male}\}$$

The participants who fall outside the grid (non-binary gender, or race that does not collapse to Black/White) are **kept in the released dataset** (they can still be useful for non-audit analyses), but are marked out of this split.

Below we apply the rule and enumerate exactly who is dropped and why.

In [4]:
race_ok   = participants["race_simplified"].isin(AUDIT_RACES)
gender_ok = participants["gender_simplified"].isin(AUDIT_GENDERS)
in_audit_eligible = race_ok & gender_ok

# Sanity-check that re-deriving the rule matches the precomputed column.
assert (in_audit_eligible == participants["audit_eligible"]).all(), (
    "Re-derived audit_eligible disagrees with the column in participants.csv — "
    "the derivation rule and the upstream pipeline have drifted."
)

n_audit = int(in_audit_eligible.sum())
clips_audit = int(participants.loc[in_audit_eligible, "n_clips"].sum())
n_dropped = n_full - n_audit
clips_dropped = clips_full - clips_audit
print(f"audit_eligible: {n_audit} participants  |  {clips_audit} clips")
print(f"  dropped from full: {n_dropped} participants ({clips_dropped} clips)")

audit_eligible: 506 participants  |  19680 clips
  dropped from full: 9 participants (328 clips)


In [5]:
print("Reasons for the audit_eligible drops:")
dropped_audit = participants[~in_audit_eligible].copy()
dropped_audit["drop_reason"] = dropped_audit.apply(
    lambda r: ", ".join(filter(None, [
        "race not Black/White"  if r["race_simplified"]   not in AUDIT_RACES   else None,
        "gender not Female/Male" if r["gender_simplified"] not in AUDIT_GENDERS else None,
    ])),
    axis=1,
)

print(dropped_audit.groupby("drop_reason").size().to_frame("n_participants"))
print()
print("Per-participant detail:")
dropped_audit[[
    "participant_id", "cohort", "race_simplified", "gender_simplified",
    "race_self_report_raw", "gender_self_report", "drop_reason",
]].sort_values(["drop_reason", "participant_id"]).reset_index(drop=True)

Reasons for the audit_eligible drops:
                        n_participants
drop_reason                           
gender not Female/Male               5
race not Black/White                 4

Per-participant detail:


,participant_id,cohort,race_simplified,gender_simplified,race_self_report_raw,gender_self_report,drop_reason
0,P0002,new_recruits,Black,Non-binary,Black or African American,Non-binary,gender not Female/Male
1,P0128,new_recruits,Black,Non-binary,Black or African American,Non-binary,gender not Female/Male
2,P0266,reconsent,Black,Non-binary,Black or African American,Non-binary,gender not Female/Male
3,P0416,reconsent,White,Non-binary,White or European American,Non-binary,gender not Female/Male
4,P0507,reconsent,White,Non-binary,White or European American,Non-binary,gender not Female/Male
5,P0035,new_recruits,Other/Unknown,Female,I prefer not to answer,Woman,race not Black/White
6,P0065,new_recruits,Other/Unknown,Male,I prefer not to answer,Man,race not Black/White
7,P0095,new_recruits,Other/Unknown,Male,I prefer not to answer,Man,race not Black/White
8,P0122,new_recruits,Other/Unknown,Male,I prefer not to answer,Man,race not Black/White


## 3. The `strict_audit` split

**Rule:** `audit_eligible` *and* the Prolific recruitment cell matches the simplified self-report.

Prolific–self-report mismatches happen when a participant whose Prolific demographic filter said e.g. "Black female" later reports something different in Qualtrics. We do not know whether each mismatch is a filter error or a genuine reclassification, so we keep these participants in `audit_eligible` and define `strict_audit` as the conservative subset where the two sources agree. The paper's primary disparity analyses run on `strict_audit`.

In [6]:
filter_ok = participants["prolific_filter_matches_self_report"]
in_strict_audit = in_audit_eligible & filter_ok

assert (in_strict_audit == participants["strict_audit_eligible"]).all(), (
    "Re-derived strict_audit_eligible disagrees with the column in participants.csv."
)

n_strict = int(in_strict_audit.sum())
clips_strict = int(participants.loc[in_strict_audit, "n_clips"].sum())
n_dropped = n_audit - n_strict
clips_dropped = clips_audit - clips_strict
print(f"strict_audit: {n_strict} participants  |  {clips_strict} clips")
print(f"  dropped from audit_eligible: {n_dropped} participants ({clips_dropped} clips)")

strict_audit: 500 participants  |  19449 clips
  dropped from audit_eligible: 6 participants (231 clips)


In [7]:
print("Participants dropped at the strict_audit stage (audit_eligible but Prolific filter ≠ self-report):")
dropped_strict = participants[in_audit_eligible & ~in_strict_audit][[
    "participant_id", "cohort", "prolific_recruitment_group",
    "race_simplified", "gender_simplified", "race_self_report_raw", "gender_self_report",
]].reset_index(drop=True)
dropped_strict

Participants dropped at the strict_audit stage (audit_eligible but Prolific filter ≠ self-report):


,participant_id,cohort,prolific_recruitment_group,race_simplified,gender_simplified,race_self_report_raw,gender_self_report
0,P0023,new_recruits,bl_fe,Black,Male,Black or African American,Man
1,P0059,new_recruits,bl_fe,White,Female,White or European American,Woman
2,P0263,reconsent,bl_fe,Black,Male,Black or African American,Man
3,P0259,reconsent,bl_fe,White,Female,White or European American,Woman
4,P0253,reconsent,bl_fe,Black,Male,Black or African American,Man
5,P0392,reconsent,wh_fe,White,Male,White or European American,Man


## 4. The `balanced` split

**Rule:** start from `strict_audit`, then uniformly sample participants within each (race × gender) cell down to the smallest cell size. Cell membership uses `race_simplified` × `gender_simplified`.

Sampling is done at the **participant level** (each selected participant contributes all of their clips). A fixed random seed (`BALANCED_SEED=42`, matching the prior release of this dataset) makes the sample deterministic and bit-for-bit reproducible.

The four `strict_audit` cells already come close to balance:

In [8]:
strict_pids_by_cell = (
    participants[in_strict_audit]
    .groupby(["race_simplified", "gender_simplified"])["participant_id"]
    .apply(list)
)
cell_sizes = strict_pids_by_cell.apply(len)
min_cell = int(cell_sizes.min())
print("strict_audit cell sizes:")
print(cell_sizes.to_string())
print(f"\nSmallest cell size → balanced target = {min_cell} per cell × 4 = {4*min_cell}")

strict_audit cell sizes:
race_simplified  gender_simplified
Black            Female               124
                 Male                 124
White            Female               126
                 Male                 126

Smallest cell size → balanced target = 124 per cell × 4 = 496


In [9]:
selected = []
for cell, pids in strict_pids_by_cell.items():
    sampled = pd.Series(pids).sample(n=min_cell, random_state=BALANCED_SEED).tolist()
    selected.extend(sampled)
selected_set = set(selected)
in_balanced = participants["participant_id"].isin(selected_set)

n_balanced = int(in_balanced.sum())
clips_balanced = int(participants.loc[in_balanced, "n_clips"].sum())
print(f"balanced: {n_balanced} participants  |  {clips_balanced} clips")
print(f"  dropped from strict_audit: {n_strict - n_balanced} participants "
      f"({clips_strict - clips_balanced} clips)")
print("\nbalanced cell sizes (verification — should be uniform):")
print(participants[in_balanced].groupby(["race_simplified", "gender_simplified"]).size().to_string())

balanced: 496 participants  |  19304 clips
  dropped from strict_audit: 4 participants (145 clips)

balanced cell sizes (verification — should be uniform):
race_simplified  gender_simplified
Black            Female               124
                 Male                 124
White            Female               124
                 Male                 124


In [10]:
print("Participants dropped at the balanced stage (in strict_audit but not sampled in their cell):")
dropped_balanced = participants[in_strict_audit & ~in_balanced][[
    "participant_id", "cohort", "race_simplified", "gender_simplified",
]].sort_values(["race_simplified", "gender_simplified", "participant_id"]).reset_index(drop=True)

print("  per-cell counts of dropped:")
print(dropped_balanced.groupby(["race_simplified", "gender_simplified"]).size().to_string())
print()
dropped_balanced

Participants dropped at the balanced stage (in strict_audit but not sampled in their cell):
  per-cell counts of dropped:
race_simplified  gender_simplified
White            Female               2
                 Male                 2



,participant_id,cohort,race_simplified,gender_simplified
0,P0143,new_recruits,White,Female
1,P0412,reconsent,White,Female
2,P0216,new_recruits,White,Male
3,P0462,reconsent,White,Male


## 5. Assemble and write `splits.csv`

Four boolean columns, one row per participant. By construction the boolean columns are nested: `in_balanced ⊆ in_strict_audit ⊆ in_audit_eligible ⊆ in_full`. We assert that invariant before writing.

In [11]:
splits = pd.DataFrame({
    "participant_id":     participants["participant_id"],
    "in_full":            in_full.values,
    "in_audit_eligible":  in_audit_eligible.values,
    "in_strict_audit":    in_strict_audit.values,
    "in_balanced":        in_balanced.values,
}).sort_values("participant_id").reset_index(drop=True)

# Nesting invariant: each split is a subset of the previous one.
assert (~splits["in_audit_eligible"] | splits["in_full"]).all()
assert (~splits["in_strict_audit"]   | splits["in_audit_eligible"]).all()
assert (~splits["in_balanced"]       | splits["in_strict_audit"]).all()

splits.to_csv(SPLITS_CSV, index=False)
print(f"Wrote {SPLITS_CSV.relative_to(REPO_ROOT)}  ({len(splits)} rows)")
splits.head()

Wrote data/metadata/splits.csv  (515 rows)


,participant_id,in_full,in_audit_eligible,in_strict_audit,in_balanced
0,P0001,True,True,True,True
1,P0002,True,False,False,False
2,P0003,True,True,True,True
3,P0004,True,True,True,True
4,P0005,True,True,True,True


## 6. Final summary

A single table cross-walking the four splits with their participant and clip counts and the headline numbers from the paper.

In [12]:
def _summary_row(name: str, mask: pd.Series) -> dict:
    return {
        "split": name,
        "n_participants": int(mask.sum()),
        "n_clips": int(participants.loc[mask.values, "n_clips"].sum()),
    }

summary = pd.DataFrame([
    _summary_row("full",            in_full),
    _summary_row("audit_eligible",  in_audit_eligible),
    _summary_row("strict_audit",    in_strict_audit),
    _summary_row("balanced",        in_balanced),
]).set_index("split")

# Headline numbers from the paper text — assert match.
expected = {
    "full":            (515, 20008),
    "audit_eligible":  (506, 19680),
    "strict_audit":    (500, 19449),
    "balanced":        (496, 19304),
}
for split_name, (exp_p, exp_c) in expected.items():
    got = summary.loc[split_name]
    assert got["n_participants"] == exp_p and got["n_clips"] == exp_c, (
        f"{split_name}: got ({got['n_participants']}, {got['n_clips']}), "
        f"expected ({exp_p}, {exp_c})"
    )
print("✓ All split sizes match the headline numbers reported in the paper.\n")
summary

✓ All split sizes match the headline numbers reported in the paper.



,n_participants,n_clips
split,,
full,515,20008
audit_eligible,506,19680
strict_audit,500,19449
balanced,496,19304


## 7. Using the splits downstream

Two patterns will cover most uses.

**Participant-level filter:**

```python
splits = pd.read_csv('data/metadata/splits.csv')
participants = pd.read_csv('data/metadata/participants.csv')
df = participants.merge(splits, on='participant_id')
df_strict = df[df['in_strict_audit']]
```

**Clip-level expansion** (no separate clip manifest needed — walk the audio tree):

In [13]:
def clips_for_split(split_col: str, splits_df: pd.DataFrame, audio_dir: Path) -> pd.DataFrame:
    """Expand a participant-level split to a clip-level DataFrame."""
    pids = set(splits_df.loc[splits_df[split_col], "participant_id"])
    rows = []
    for cohort_dir in audio_dir.iterdir():
        if not cohort_dir.is_dir():
            continue
        for pid_dir in cohort_dir.iterdir():
            if pid_dir.name not in pids:
                continue
            for wav in pid_dir.glob("*.wav"):
                rows.append({
                    "participant_id": pid_dir.name,
                    "cohort": cohort_dir.name,
                    "clip_id": wav.stem,
                    "audio_path": str(wav.relative_to(audio_dir.parent.parent)),
                })
    return pd.DataFrame(rows).sort_values(["participant_id", "clip_id"]).reset_index(drop=True)


# Spot-check: balanced should expand to 19,304 clips across 496 participants.
balanced_clips = clips_for_split("in_balanced", splits, AUDIO_DIR)
print(f"balanced expanded to clip-level: {len(balanced_clips)} clips "
      f"across {balanced_clips['participant_id'].nunique()} participants")
balanced_clips.head()

balanced expanded to clip-level: 19304 clips across 496 participants


,participant_id,cohort,clip_id,audio_path
0,P0001,new_recruits,P0001_mic_check,data/audio/new_recruits/P0001/P0001_mic_check.wav
1,P0001,new_recruits,P0001_q01,data/audio/new_recruits/P0001/P0001_q01.wav
2,P0001,new_recruits,P0001_q04,data/audio/new_recruits/P0001/P0001_q04.wav
3,P0001,new_recruits,P0001_q05,data/audio/new_recruits/P0001/P0001_q05.wav
4,P0001,new_recruits,P0001_q07,data/audio/new_recruits/P0001/P0001_q07.wav
